# KV Cache Compression: Eviction and Quantization

The full code of [KV Cache Compression: Eviction and Quantization](https://residual-stream.lvtan71.workers.dev/posts/kv-cache-compression/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/kv-cache-compression) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/kv-cache-compression.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 28 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 28 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "kv-cache-compression"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/kv-cache-compression


## `budget.py`

The KV cache's bytes per token and size for one 32k-token sequence, for Llama-2-13B, under fewer KV heads, 4- and 2-bit quantization, keeping a quarter of the tokens, and all three combined.

In [2]:
!{sys.executable} budget.py

                                            bytes per token  one 32k sequence  vs baseline
BF16, every token (baseline)                      819,200 B         25.00 GiB      100.0%
if it had 8 KV heads (GQA)                        163,840 B          5.00 GiB       20.0%
4-bit K and V                                     256,000 B          7.81 GiB       31.2%
2-bit K and V (KIVI)                              153,600 B          4.69 GiB       18.8%
keep a quarter of the tokens                      204,800 B          6.25 GiB       25.0%
GQA-8 + 4-bit + a quarter of the tokens            12,800 B          0.39 GiB        1.6%


The post printed:

```text
                                            bytes per token  one 32k sequence  vs baseline
BF16, every token (baseline)                      819,200 B         25.00 GiB      100.0%
if it had 8 KV heads (GQA)                        163,840 B          5.00 GiB       20.0%
4-bit K and V                                     256,000 B          7.81 GiB       31.2%
2-bit K and V (KIVI)                              153,600 B          4.69 GiB       18.8%
keep a quarter of the tokens                      204,800 B          6.25 GiB       25.0%
GQA-8 + 4-bit + a quarter of the tokens            12,800 B          0.39 GiB        1.6%
```

## `measure.py`

Qwen2.5-0.5B's loss in bits per token on the last 1,024 tokens of eight 2,048-token windows of held-out blog text, with the full cache, with eviction (recent windows, with and without 4 sink tokens), and with the cached keys and values quantized (8/4/2 bits, per token or per channel, KIVI's full-precision recent window), a control with the key bias removed before rounding, and the size of the key bias in two layers.

In [3]:
!{sys.executable} measure.py

full cache: 3.063 bits per token, 8 windows
eviction: keep


  the last 512 tokens                          4.957  (+1.894 ± 0.031)


  4 sink tokens + the last 508                 3.235  (+0.172 ± 0.011)


  the last 256 tokens                          5.254  (+2.191 ± 0.034)


  4 sink tokens + the last 252                 3.437  (+0.374 ± 0.016)


...

  2-bit, keys per channel                      4.589  (+1.526 ± 0.024)


  2-bit, keys per channel, last 128 exact (KIVI) 3.388  (+0.325 ± 0.013)
control: keys per token with the key bias taken out first (before RoPE)


  4-bit keys, per token, bias removed          3.146  (+0.083 ± 0.006)


  2-bit keys, per token, bias removed          8.043  (+4.980 ± 0.046)


layer  0 keys: largest bias 130.0; a token's 64 channels span 243.7 on average,  4.4 without the bias; a channel varies from token to token by 0.40
layer 12 keys: largest bias   4.4; a token's 64 channels span  11.8 on average,  7.4 without the bias; a channel varies from token to token by 0.75


The post printed:

```text
full cache: 3.063 bits per token, 8 windows
eviction: keep
  the last 512 tokens                          4.957  (+1.894 ± 0.031)
  4 sink tokens + the last 508                 3.235  (+0.172 ± 0.011)
  the last 256 tokens                          5.254  (+2.191 ± 0.034)
  4 sink tokens + the last 252                 3.437  (+0.374 ± 0.016)
  the last 128 tokens                          5.735  (+2.672 ± 0.039)
  4 sink tokens + the last 124                 3.689  (+0.626 ± 0.021)
quantization of the cache (keys after RoPE):
  8-bit, per token                             3.112  (+0.049 ± 0.006)
  4-bit, per token                             5.912  (+2.849 ± 0.036)
  4-bit, only values, per token                3.074  (+0.011 ± 0.002)
  4-bit, only keys, per token                  5.937  (+2.874 ± 0.036)
  4-bit, keys per channel                      3.087  (+0.024 ± 0.003)
  2-bit, per token                             11.506  (+8.443 ± 0.059)
  2-bit, keys per channel                      4.592  (+1.529 ± 0.024)
  2-bit, keys per channel, last 128 exact (KIVI) 3.388  (+0.325 ± 0.013)
control: keys per token with the key bias taken out first (before RoPE)
  4-bit keys, per token, bias removed          3.136  (+0.073 ± 0.006)
  2-bit keys, per token, bias removed          8.048  (+4.985 ± 0.047)
layer  0 keys: largest bias 130.0; a token's 64 channels span 243.7 on average,  4.4 without the bias; a channel varies from token to token by 0.40
layer 12 keys: largest bias   4.4; a token's 64 channels span  11.8 on average,  7.4 without the bias; a channel varies from token to token by 0.75
```